# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema
dataset = mlc.Dataset(croissant_url)

# Print metadata summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print('\nKeywords:', getattr(metadata, 'keywords', None))
print('\nFAIR2 CiteAs:', getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', None)))
print('\nCroissant JSON-LD Schema URL:', croissant_url)

## 2. Data Overview
Review available record sets, fields, and their IDs. All references will use entities' `@id`.

In [ ]:
# List available record sets and their fields. Each entity is referenced by its `@id`.
print('Record Sets in dataset:')
record_sets = []
for rs in dataset.record_sets():
    print(f"- name: {rs.name}  (@id: {rs.id})")
    record_sets.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()

if not record_sets:
    print('No record sets found in the top-level metadata; you may need to explore via .record_sets().')

Let's demonstrate listing records from the first available record set, if present.

In [ ]:
# Display the first 2 records for the primary record set as an example
if record_sets:
    primary_record_set_id = record_sets[0]
    print(f"\nSample records from Record Set @id = {primary_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=primary_record_set_id)):
        print(json.dumps(record, indent=2))
        if i >= 1:  # show only the first 2 records
            break
else:
    print("No record set found; please check the dataset structure.")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. All names and access patterns use record set and field `@id`s from above. Here we use all detected record sets.

In [ ]:
# Extract all available record sets to pandas DataFrames by their `@id`
dataframes = {}

if record_sets:
    for rs_id in record_sets:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
        else:
            dataframes[rs_id] = pd.DataFrame()
else:
    print('No record sets found. This dataset may not have any tabular data or may use a custom structure.')

# Show the columns of the first table and display its head
if record_sets and dataframes[record_sets[0]].shape[0] > 0:
    print(f"\nColumns in DataFrame for record set {record_sets[0]}:")
    print(dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())
else:
    print('No rows found in main record set.')

## 4. Exploratory Data Analysis (EDA)
Here we select a numeric field for analysis, filter values, normalize the numeric column, and group data by a categorical field, referencing all columns by their `@id`.

You may adjust `numeric_field_id` and `group_field_id` as found appropriate for the dataset.

In [ ]:
# Choose a record set and fields for demonstration (adjust based on listing above)
if record_sets and not dataframes[record_sets[0]].empty:
    # Try to guess a numeric field by dtype
    df = dataframes[record_sets[0]]
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Use first numeric column
        print(f"Selected numeric field: {numeric_field_id}")

        # Try to guess a group/categorical field
        nonnum_cols = [c for c in df.columns if c not in numeric_cols]
        group_field_id = None
        if nonnum_cols:
            # pick a column with low unique count (<20) to group by
            for c in nonnum_cols:
                if df[c].nunique() < 20:
                    group_field_id = c
                    break

        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != 'O' else 0  # Or any arbitrary value
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows\n")
        display(filtered_df.head())

        normcol = f"{numeric_field_id}_normalized"
        filtered_df[normcol] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) \
                                       / filtered_df[numeric_field_id].std()
        print(f"\nFirst rows with normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, normcol]].head())

        if group_field_id:
            print(f"\nMean-normalized {numeric_field_id} grouped by {group_field_id}:")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print('No suitable categorical field found for grouping.')
    else:
        print('No numeric fields found in the main record set.')
else:
    print('No data found for EDA.')

## 5. Visualization
Let's plot the distribution of one numeric variable if available, using only references via field `@id` (column name in DataFrame).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and record_sets[0] in dataframes and not dataframes[record_sets[0]].empty:
    df = dataframes[record_sets[0]]
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_cols:
        field = numeric_cols[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {field}")
        plt.xlabel(field)
        plt.show()

        # If a group/categorical field exists, show mean by group
        cat_cols = [c for c in df.columns if (df[c].dtype == object or df[c].dtype.name == 'category') and df[c].nunique() < 20]
        if cat_cols:
            cfield = cat_cols[0]
            plt.figure(figsize=(8, 4))
            sns.barplot(x=cfield, y=field, data=df, errorbar='ci')
            plt.title(f"Mean {field} by {cfield}")
            plt.ylabel(field)
            plt.xlabel(cfield)
            plt.show()
    else:
        print('No numeric columns available for visualization.')
else:
    print('No data available for visualization.')

## 6. Conclusion
In this notebook, you:
- Explored a Croissant metadata-based dataset using the `mlcroissant` package
- Inspected all record sets and fields using their `@id`
- Loaded data into pandas DataFrames for downstream processing
- Performed exploratory and summary analyses referencing fields via `@id`
- Produced basic visualizations to summarize the dataset

This provides a foundation for more complex processing (e.g. feature engineering or publication-ready analysis) using FAIR data packages and Croissant-compliant datasets.